- Case Study 1: Lights, Camera, SQL!

- Objective

The objective of this analysis is to determine what defines the “best” movie investment using SQL queries on the provided movie database.

For this case study, we define “best” primarily in terms of Return on Investment (ROI), calculated as:

ROI = (Worldwide − Budget) / Budget

We also analyze genre performance and domestic vs international revenue patterns.

In [2]:
import sqlite3
import pandas as pd


## 1. Connecting to the Database

We begin by connecting to the SQLite database and verifying the available tables.

In [3]:
conn = sqlite3.connect("Data/movie.sqlite")


We establish a connection to the SQLite database so we can execute SQL queries against the movie dataset.

In [4]:
tables_query = """
SELECT name 
FROM sqlite_master 
WHERE type='table';
"""


In [7]:
tables_df = pd.read_sql(tables_query, conn)
tables_df


,name
0,IMDB
1,earning
2,genre


In [ ]:
The database contains three main tables:

- IMDB (movie metadata such as title, rating, budget)
- earning (domestic and worldwide revenue)
- genre (movie genre classification)

These tables will be joined using Movie_id.

In [11]:
imdb_preview = """
SELECT *
FROM IMDB
LIMIT 5;
"""
imdb_df = pd.read_sql(imdb_preview, conn)
imdb_df


,Movie_id,Title,Rating,TotalVotes,MetaCritic,Budget,Runtime,CVotes10,CVotes09,CVotes08,...,Votes3044,Votes3044M,Votes3044F,Votes45A,Votes45AM,Votes45AF,VotesIMDB,Votes1000,VotesUS,VotesnUS
0,36809,12 Years a Slave (2013),8.1,496092,96,20000000.0,134 min,75556,126223,161460,...,8.0,7.9,8.0,7.8,7.8,8.1,8.0,7.7,8.3,8.0
1,30114,127 Hours (2010),7.6,297075,82,18000000.0,94 min,28939,44110,98845,...,7.5,7.5,7.5,7.3,7.3,7.5,7.6,7.0,7.7,7.6
2,37367,50/50 (2011),7.7,283935,72,8000000.0,100 min,28304,47501,99524,...,7.6,7.6,7.6,7.4,7.4,7.5,7.4,7.0,7.9,7.6
3,49473,About Time (2013),7.8,225412,,12000000.0,123 min,38556,43170,70850,...,7.6,7.6,7.7,7.6,7.5,7.8,7.7,6.9,7.8,7.7
4,14867,Amour (2012),7.9,76121,94,8900000.0,127 min,11093,15944,22942,...,7.7,7.7,7.9,7.9,7.8,8.1,6.6,7.2,7.9,7.8


In [ ]:
The IMDB table includes key variables such as Title, Rating, Budget, and TotalVotes.

The Budget column is especially important for ROI calculation.

In [12]:
earning_preview = """
SELECT *
FROM earning
LIMIT 5;
"""
earning_df = pd.read_sql(earning_preview, conn)
earning_df


,Movie_id,Domestic,Worldwide
0,36809,56671993,187733202.0
1,30114,18335230,60738797.0
2,37367,35014192,39187783.0
3,49473,15322921,87100449.0
4,14867,6739492,19839492.0


In [ ]:
The earning table contains Domestic and Worldwide revenue values.

Worldwide revenue will be used to calculate profit and ROI.

In [13]:
genre_preview = """
SELECT *
FROM genre
LIMIT 5;
"""
genre_df = pd.read_sql(genre_preview, conn)
genre_df


,Movie_id,genre
0,36809,Biography
1,30114,Adventure
2,37367,Comedy
3,49473,Comedy
4,14867,Drama


In [ ]:
The genre table associates each Movie_id with a genre classification.

This allows us to analyze investment performance at the genre level.

In [14]:
financial_query = """
SELECT 
    m.Movie_id,
    m.Title,
    m.Budget,
    e.Domestic,
    e.Worldwide,
    (e.Worldwide - m.Budget) AS Profit,
    ROUND((e.Worldwide - m.Budget) * 1.0 / m.Budget, 2) AS ROI
FROM IMDB m
JOIN earning e
    ON m.Movie_id = e.Movie_id
WHERE m.Budget IS NOT NULL
AND e.Worldwide IS NOT NULL
AND m.Budget > 0
ORDER BY ROI DESC;
"""


In [15]:
financial_df = pd.read_sql(financial_query, conn)
financial_df.head(10)


,Movie_id,Title,Budget,Domestic,Worldwide,Profit,ROI
0,35832,The King's Speech (2010),15000000.0,135453143,414211549.0,399211549.0,26.61
1,42727,The Fault in Our Stars (2014),12000000.0,124872350,307166834.0,295166834.0,24.60
2,43358,Black Swan (2010),13000000.0,106954678,329398046.0,316398046.0,24.34
3,45458,The Imitation Game (2014),14000000.0,91125683,233555708.0,219555708.0,15.68
4,31069,La La Land (2016),30000000.0,151101803,445669679.0,415669679.0,13.86
5,38262,Whiplash (2014),3300000.0,13092000,48982041.0,45682041.0,13.84
6,18506,Deadpool (2016),58000000.0,363070709,783112979.0,725112979.0,12.50
7,47616,Lion (2016),12000000.0,51728731,140302754.0,128302754.0,10.69
8,17294,Silver Linings Playbook (2012),21000000.0,132092958,236412453.0,215412453.0,10.26
9,36798,Boyhood (2014),4000000.0,25352281,44495281.0,40495281.0,10.12


The ROI calculation shows that high returns are not necessarily associated with the largest budgets.

This highlights the importance of investment efficiency rather than scale alone.

In [16]:
genre_query = """
SELECT 
    g.genre,
    COUNT(*) AS Num_Movies,
    ROUND(AVG((e.Worldwide - m.Budget) * 1.0 / m.Budget), 2) AS Avg_ROI
FROM IMDB m
JOIN earning e
    ON m.Movie_id = e.Movie_id
JOIN genre g
    ON m.Movie_id = g.Movie_id
WHERE m.Budget IS NOT NULL
AND e.Worldwide IS NOT NULL
AND m.Budget > 0
GROUP BY g.genre
ORDER BY Avg_ROI DESC;
"""


In [17]:
genre_df = pd.read_sql(genre_query, conn)
genre_df


,genre,Num_Movies,Avg_ROI
0,Music,3,10.03
1,Biography,21,6.73
2,Musical,1,6.24
3,,41,5.97
4,Thriller,14,5.70
5,Romance,13,5.66
6,History,6,5.49
7,Fantasy,7,5.07
8,Drama,77,5.07
9,Western,2,4.45


In [ ]:
Genre-level analysis reveals that some categories consistently outperform others in terms of average ROI.

Genres with larger sample sizes provide more reliable investment insights.

In [4]:
genre_query_min10 = """
SELECT 
    g.genre,
    COUNT(*) AS Num_Movies,
    ROUND(AVG((e.Worldwide - m.Budget) * 1.0 / m.Budget), 2) AS Avg_ROI
FROM IMDB m
JOIN earning e
    ON m.Movie_id = e.Movie_id
JOIN genre g
    ON m.Movie_id = g.Movie_id
WHERE m.Budget IS NOT NULL
AND e.Worldwide IS NOT NULL
AND m.Budget > 0
GROUP BY g.genre
HAVING COUNT(*) >= 10
ORDER BY Avg_ROI DESC;
"""
genre_min10_df = pd.read_sql(genre_query_min10, conn)
genre_min10_df

,genre,Num_Movies,Avg_ROI
0,Biography,21,6.73
1,,41,5.97
2,Thriller,14,5.70
3,Romance,13,5.66
4,Drama,77,5.07
5,Comedy,31,4.13
6,Adventure,43,3.53
7,Action,33,3.47
8,Animation,13,3.29
9,Crime,11,3.21


In [ ]:
Among genres with at least 10 films, **Biography** has the highest average ROI (6.73), followed by **Thriller** (5.70) and **Romance** (5.66).

While Drama has the largest sample size (77 films), its average ROI (5.07) is lower than the top-performing categories.

This suggests that although Drama offers stable volume, certain specialized genres such as Biography may provide stronger investment efficiency.

In [5]:
market_query = """
SELECT
    ROUND(AVG(Domestic),0) AS Avg_Domestic,
    ROUND(AVG(Worldwide - Domestic),0) AS Avg_International
FROM earning
WHERE Domestic IS NOT NULL
AND Worldwide IS NOT NULL;
"""
market_df = pd.read_sql(market_query, conn)
market_df

,Avg_Domestic,Avg_International
0,136029365.0,204348647.0


International revenue exceeds domestic revenue on average.

This suggests that global market performance plays a significant role in defining a successful investment.

In [ ]:
---

# Final Conclusion: What Defines the “Best” Movie Investment?

Based on our SQL analysis:

1. High ROI films are not necessarily high-budget films, indicating that capital efficiency is more important than scale.
2. Biography, Thriller, and Romance genres generate the strongest average ROI among genres with sufficient sample size.
3. International revenue exceeds domestic revenue on average, highlighting the importance of global market performance.

Overall, the “best” movie investment is one that maximizes return efficiency while leveraging international market appeal, rather than simply relying on high production budgets.